In [106]:
#!pip install rank-bm25
#!pip install langchain-chroma
#!pip install langchain-huggingface
#!pip install -U langchain langchain-openai

# MultiHopRag

In [1]:
import os, re, zipfile, torch,  json
from typing import List, Dict

import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize
from google.colab import drive, userdata

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langsmith import traceable

# API Key 설정
try:
    os.environ["LANGCHAIN_API_KEY"] = userdata.get('langgrpah')
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGCHAIN_PROJECT"] = "h4s1"

    # OpenRouter 키 (변수명 확인 필요)
    openrouter_key = userdata.get('OPENROUTER')
except Exception as e:
    print(f"Key 설정 오류: {e}")

print("환경 설정 완료")

from langchain_core.runnables import (
    RunnableLambda,
    RunnableBranch,
    RunnablePassthrough
)

from langchain_openai import ChatOpenAI


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


환경 설정 완료


In [5]:
ZIP_PATH = "/content/drive/MyDrive/chroma_db_bge_m3-20260114T091037Z-3-001.zip"

DB_PATH = "/content/drive/MyDrive/chroma_db_bge_m3"

os.makedirs(DB_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DB_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=openrouter_key,
    temperature=0,
    seed=2026
)

embedder = SentenceTransformer("BAAI/bge-m3", device=device)

def load_all_docs():
    real_path = DB_PATH
    for root, _, files in os.walk(DB_PATH):
        if "chroma.sqlite3" in files:
            real_path = root
            break

    db = Chroma(
        persist_directory=real_path,
        embedding_function=embeddings,
        collection_name="multihop_rag",
    )

    data = db.get(include=["documents", "metadatas"])
    return data["documents"], data["metadatas"]

all_texts, all_metas = load_all_docs()


### **Question Type Classification**

In [6]:
drive.mount("/content/drive/model", force_remount=True)

QTYPE_MODEL_PATH = "/content/drive/model/MyDrive/qtype_model"
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

qtype_tokenizer = AutoTokenizer.from_pretrained(QTYPE_MODEL_PATH)
qtype_model = AutoModelForSequenceClassification.from_pretrained(QTYPE_MODEL_PATH)
qtype_model.to(device)
qtype_model.eval()

Mounted at /content/drive/model


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [7]:
LABEL_MAP = {
    0: "comparison_query",
    1: "inference_query",
    2: "temporal_query",
    3: "null_query"
}

def predict_question_type(question: str) -> str:
    inputs = qtype_tokenizer(
        question,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = qtype_model(**inputs).logits
        pred = torch.argmax(logits, dim=-1).item()

    return LABEL_MAP[pred]

### **BM25 Retrieval**

In [8]:
def tokenize(text: str):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(t) for t in all_texts])

def bm25_retrieve(query: str, k=6) -> List[Document]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

    return [
        Document(
            page_content=all_texts[i],
            metadata=all_metas[i]
        )
        for i in top_idx
    ]


### **Hybrid Dense Reranking**

In [10]:
import numpy as np

def bm25_retrieve_with_score(query, k=20):
    docs = bm25_retrieve(query, k)

    # BM25 점수 대신 "순위 기반 가중치" 사용
    return [(doc, 1 / (i + 1)) for i, doc in enumerate(docs)]


def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def hybrid_rerank(query, bm25_results, alpha=0.4, top_k=5):
    query_emb = embedder.encode(query, normalize_embeddings=True)

    scored = []
    for doc, bm25_score in bm25_results:
        doc_emb = embedder.encode(
            doc.page_content,
            normalize_embeddings=True
        )
        dense_score = cosine_sim(query_emb, doc_emb)
        final_score = alpha * bm25_score + (1 - alpha) * dense_score
        scored.append((doc, final_score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scored[:top_k]]



### **Evidence Extraction (LLM)**

In [11]:
def extract_evidence(question: str, docs: List[Document]) -> Dict:
    doc_blocks = ""
    for i, d in enumerate(docs, 1):
        doc_blocks += f"""
[Document {i}]
Source: {d.metadata.get('source')}
Title: {d.metadata.get('title')}
Content:
{d.page_content[:2000]}
"""

    prompt = f"""
You are an evidence extraction system.

Task:
- For EACH document, extract AT MOST ONE sentence that is RELEVANT to ANY PART of the question.
- The sentence does NOT need to fully answer the question by itself.
- It is sufficient if the sentence provides PARTIAL information useful for later reasoning.
- Copy the sentence verbatim from the document.
- If absolutely nothing is relevant, return null.

This task is NOT to answer the question, but to collect building blocks for multi-document reasoning.

For each extracted fact:
- Set "source" to the Source field shown above the document.
- Set "url" to the URL field shown above the document.
- Do NOT infer or guess.

- Return at most ONE evidence item per document.
- Include the document_id for each extracted fact.

Question:
{question}

Documents:
{doc_blocks}

Output JSON format:
{{
  "evidence": [
    {{
      "document_id": 1,
      "fact": "...",
      "source": "...",
      "url": "..."
    }}
  ]
}}

Rules:
- Do NOT answer the question.
- Do NOT paraphrase.
- Evidence list may be empty.
"""

    return llm.invoke(prompt).content


### **Question-type-aware Reasoning (LLM)**

In [12]:
def final_reasoning(question: str, evidence_json: str) -> str:
    q_type = predict_question_type(question)

    # 1️⃣ null_query → 무조건 증거 불충분
    if q_type == "null_query":
        return (
            "Answer: Insufficient Information\n"
            "Reasoning: The available evidence is insufficient to answer the question."
        )

    # 2️⃣ inference_query → entity 추론
    if q_type == "inference_query":
        rules = """
Rules:
- Use ONLY the evidence.
- Identify the SINGLE common entity, individual, organization, or term that satisfies ALL parts of the question.
- Do NOT answer Yes or No.
- The answer must be a short entity name or common noun.
- If no such entity can be identified, answer: Insufficient Information.
"""
        answer_formats = """
Answer format:
Answer: <entity name>
Reasoning: <1–2 sentences>
"""

    else:  # comparison_query or temporal_query
        rules = """
Rules:
- Use ONLY the evidence.
- Determine whether the evidence from different articles is CONSISTENT or INCONSISTENT with each other.
- Articles ALIGN if they support a compatible or complementary understanding of the topic, even if they discuss different aspects.
- Articles DO NOT align only if they contradict each other.
- Answer "Yes" if the articles align.
- Answer "No" if they contradict.
- Answer "Insufficient Information" only if the relationship cannot be determined.
"""
        answer_formats = """
Answer format:
Answer: Yes / No / Insufficient Information
Reasoning: <1–2 sentences>
"""

    prompt = f"""
You are a reasoning system.

Question:
{question}

Evidence:
{evidence_json}

{rules}

{answer_formats}
"""

    return llm.invoke(prompt).content.strip()


In [13]:
def safe_extract_evidence(raw_evidence):
    """
    raw_evidence가
    - None
    - JSON 문자열
    - dict
    어떤 경우든 안전하게 evidence list를 반환
    """
    if raw_evidence is None:
        return []

    if isinstance(raw_evidence, dict):
        return raw_evidence.get("evidence", [])

    if isinstance(raw_evidence, str):
        try:
            parsed = json.loads(raw_evidence)
            return parsed.get("evidence", [])
        except Exception:
            return []

    return []

In [14]:
def build_multihop_rag_chain():
    return (
        # 0. 입력 정규화 (str / dict 모두 허용)
        RunnableLambda(
            lambda x: x if isinstance(x, dict) else {"query": x}
        )

        # 1. question_type 추가
        | RunnableLambda(lambda x: {
            **x,
            "question_type": predict_question_type(x["query"])
        })

        # 2. retrieval (기존 로직 그대로)
        | RunnableLambda(lambda x: {
            **x,
            "bm25_docs": bm25_retrieve_with_score(x["query"])
        })
        | RunnableLambda(lambda x: {
            **x,
            "docs": hybrid_rerank(x["query"], x["bm25_docs"], alpha=0.4, top_k=5)
        })

        # 3. evidence extraction
        | RunnableLambda(lambda x: {
            **x,
            "raw_evidence": extract_evidence(x["query"], x["docs"])
        })

        # 4. evidence_list 구성 (메타데이터 매핑만)
        | RunnableLambda(lambda x: {
            **x,
            "evidence_list": [
                {
                    "author": x["docs"][e["document_id"] - 1].metadata.get("author"),
                    "category": x["docs"][e["document_id"] - 1].metadata.get("category"),
                    "fact": e.get("fact"),
                    "published_at": x["docs"][e["document_id"] - 1].metadata.get("published_at"),
                    "source": x["docs"][e["document_id"] - 1].metadata.get("source"),
                    "title": x["docs"][e["document_id"] - 1].metadata.get("title"),
                    "url": x["docs"][e["document_id"] - 1].metadata.get("url"),
                }
                for e in safe_extract_evidence(x.get("raw_evidence"))
                if (
                    isinstance(e, dict)
                    and "document_id" in e
                    and 0 <= (e["document_id"] - 1) < len(x["docs"])
                )
            ]
        })

        # 5. final reasoning (기존 함수 그대로)
        | RunnableLambda(lambda x: {
            "query": x["query"],
            "question_type": x["question_type"],
            "answer": final_reasoning(
                x["query"],
                {
                  "evidence": [
                    {
                      "fact": ev["fact"],
                      "source": ev["source"],
                      "published_at": ev["published_at"]
                    }
                    for ev in x["evidence_list"]
                ]
              }
            ),
            "evidence_list": x["evidence_list"]
        })
    )


In [ ]:
USER_QUERY = "Between the TechCrunch report on Meta's moderation bias problem suppressing Palestinian voices published on October 19, 2023, and the TechCrunch report on Meta's perfect compliance with the Children’s Online Privacy Protection Act published on November 27, 2023, was there a change in the nature of issues reported concerning Meta's platform practices?"

In [15]:
chain = build_multihop_rag_chain()

result = chain.invoke(
    "Does 'The Independent - Life and Style' article suggesting Prince William's emotional state regarding Princess Diana's death align with the same publication's depiction of the events leading up to her death in 'The Crown season six'?"
)

result

{'query': "Does 'The Independent - Life and Style' article suggesting Prince William's emotional state regarding Princess Diana's death align with the same publication's depiction of the events leading up to her death in 'The Crown season six'?",
 'question_type': 'comparison_query',
 'answer': "Answer: Yes  \nReasoning: The evidence from both articles discusses the events leading up to Princess Diana's death and the emotional context surrounding it, indicating a consistent portrayal of the timeline and circumstances. The articles complement each other by providing different aspects of the same historical events.",
 'evidence_list': [{'author': 'Isobel Lewis',
   'category': 'entertainment',
   'fact': 'The first part of season six takes place in the summer of 1997, and sees Prince Charles (Dominic West) building on his relationship with Camilla Parker Bowles (Olivia Williams), while Princess Diana (Elizabeth Debicki) sparks up a romance with film producer Dodi Fayed (Khalid Abdalla).'

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from datasets import load_dataset

# QA 데이터셋 로드
try:
    qa_df
except NameError:
    print("QA 데이터셋 로드 중...")
    ds = load_dataset("yixuantt/MultiHopRAG", "MultiHopRAG", split="train")
    qa_df = ds.to_pandas()

# 층화 추출 (Stratified Sampling) - 비율 맞춰서 50개 뽑기
print(f"전체 데이터 개수: {len(qa_df)}")
print("Question Type별 비율에 맞춰 50개 샘플링 중...")

# sklearn을 사용하여 비율 유지하며 추출
sampled_df, _ = train_test_split(
    qa_df,
    train_size=50,
    stratify=qa_df['question_type'],
    random_state=42 # 재현성을 위해 시드 고정
)

print(f"추출된 샘플 개수: {len(sampled_df)}")
print(sampled_df['question_type'].value_counts()) # 타입별 개수 확인

QA 데이터셋 로드 중...
전체 데이터 개수: 2556
Question Type별 비율에 맞춰 50개 샘플링 중...
추출된 샘플 개수: 50
question_type
comparison_query    17
inference_query     16
temporal_query      11
null_query           6
Name: count, dtype: int64


In [17]:
from tqdm import tqdm
import pandas as pd

results = []

for _, row in tqdm(sampled_df.iterrows(), total=len(sampled_df)):
    query = row["query"]

    rag_out = chain.invoke(query)   # ← 각자 자기 RAG 코드

    results.append({
        "query": query,
        "question_type": row["question_type"],      # gold
        "answer": row["answer"],                    # gold answer
        "evidence_list": row["evidence_list"],      # gold evidence
        "RAG_answer": rag_out.get("answer"),
        "RAG_evidence_list": rag_out.get("evidence_list")
    })

final_eval_df = pd.DataFrame(results)

final_eval_df.to_csv("/content/drive/MyDrive/rag_eval.csv", index = False, encoding = 'utf-8-sig' )


100%|██████████| 50/50 [08:23<00:00, 10.07s/it]


In [18]:
pd.read_csv("/content/drive/MyDrive/rag_eval.csv")

,query,question_type,answer,evidence_list,RAG_answer,RAG_evidence_list
0,"Does the TechCrunch article suggest that ""Peop...",comparison_query,Yes,"[{'author': 'Sarah Perez', 'category': 'techno...",Answer: Yes \nReasoning: The TechCrunch artic...,"[{'author': 'Sarah Perez', 'category': 'techno..."
1,Who is the individual targeted by Attorney Gen...,inference_query,Donald Trump,"[{'author': 'Michael R. Sisak, The Associated ...",Answer: Donald Trump \nReasoning: The evidenc...,"[{'author': 'Michael R. Sisak, The Associated ..."
2,"Between the report by The Age on October 22, 2...",temporal_query,Yes,"[{'author': 'Kyle Wiggers', 'category': 'techn...",Answer: Yes \nReasoning: Both articles highli...,"[{'author': 'Sarah Perez', 'category': 'techno..."
3,Who is the individual under 30 who was once co...,inference_query,Sam Bankman-Fried,"[{'author': 'Elizabeth Lopatto', 'category': '...",Answer: Sam Bankman-Fried \nReasoning: Sam Ba...,"[{'author': 'Elizabeth Lopatto', 'category': '..."
4,Does the TechCrunch article report on new hiri...,comparison_query,no,"[{'author': 'Alyssa Stringer', 'category': 'te...",Answer: No \nReasoning: The TechCrunch articl...,"[{'author': 'Jessica Conditt', 'category': 'te..."
5,Does the TechCrunch article discussing Meta's ...,comparison_query,Yes,"[{'author': 'Morgan Sung', 'category': 'techno...",Answer: Yes \nReasoning: The articles discuss...,"[{'author': 'Morgan Sung', 'category': 'techno..."
6,"Which company, according to Eddy Cue, had no v...",inference_query,Google,"[{'author': 'David Pierce', 'category': 'techn...",Answer: Google \nReasoning: The evidence indi...,"[{'author': 'Sarah Perez', 'category': 'techno..."
7,Between the article from 'The Independent - Li...,temporal_query,Yes,"[{'author': 'Chelsea Ritschel', 'category': 'e...",Answer: Yes \nReasoning: Both articles portra...,"[{'author': 'Amber Raiken', 'category': 'enter..."
8,Considering the information from an article in...,null_query,Insufficient information.,[],Answer: Insufficient Information\nReasoning: T...,"[{'author': 'None', 'category': 'entertainment..."
9,Does the article from 'The Independent - Life ...,comparison_query,Yes,"[{'author': 'Chelsea Ritschel', 'category': 'e...",Answer: Yes \nReasoning: The article from 'Th...,"[{'author': 'Chelsea Ritschel', 'category': 'e..."


In [19]:
csv_path = "/content/drive/MyDrive/rag_eval.csv"
df = pd.read_csv(csv_path)

df.head()

def clean_rag_answer(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    # Case 1: Answer: xxx\nReasoning: ...
    match = re.search(r"Answer:\s*(.*?)\s*(?:\n|$)", text, re.IGNORECASE)
    if match:
        return match.group(1).strip()

    # Case 2: 이미 Yes / No / Entity 만 있는 경우
    return text

In [20]:
df["RAG_answer"] = df["RAG_answer"].apply(clean_rag_answer)

df[["answer", "RAG_answer"]].head(10)

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("RAG_answer 전처리 완료 및 저장 완료")


RAG_answer 전처리 완료 및 저장 완료


In [21]:
# @title 평가지표
import pandas as pd
import ast
import re
import string
from collections import Counter
# import os
# from google.colab import drive, userdata


# # 1. 구글 드라이브 마운트 (데이터가 드라이브에 있으므로 필수)
# if not os.path.exists('/content/drive'):
#     drive.mount('/content/drive')

# 1. 파일 로드 (경로는 사용자 환경에 맞게 유지)
df = pd.read_csv("/content/drive/MyDrive/rag_eval.csv", encoding="utf-8-sig")

# ------------------------------------------------------------------
# [Helper] 파싱 함수
# ------------------------------------------------------------------
def parse_list_robust(x):
    if not isinstance(x, str): return []
    try:
        return ast.literal_eval(x)
    except:
        pass
    fixed_str = re.sub(r'\}\s*\{', '}, {', x)
    try:
        return ast.literal_eval(fixed_str)
    except:
        return []

df['evidence_list'] = df['evidence_list'].apply(parse_list_robust)
df['RAG_evidence_list'] = df['RAG_evidence_list'].apply(parse_list_robust)

# ------------------------------------------------------------------
# [Metric 1] Retrieval Metrics (Hit, MRR, MAP)
# ------------------------------------------------------------------
def calculate_retrieval_metrics(row, k=5):
    gold_urls = set([item.get('url') for item in row['evidence_list'] if item.get('url')])
    raw_retrieved_urls = [item.get('url') for item in row['RAG_evidence_list'] if item.get('url')]

    # 중복 제거 (순서 유지)
    retrieved_urls = []
    seen = set()
    for url in raw_retrieved_urls:
        if url not in seen:
            retrieved_urls.append(url)
            seen.add(url)
    retrieved_urls = retrieved_urls[:k]

    # Hit@K
    hit = 1 if not gold_urls.isdisjoint(retrieved_urls) else 0

    # MRR@K
    mrr = 0
    for i, url in enumerate(retrieved_urls):
        if url in gold_urls:
            mrr = 1 / (i + 1)
            break

    # MAP@K
    num_gold = len(gold_urls)
    if num_gold == 0:
        ap = 0
    else:
        hits = 0
        sum_precisions = 0
        for i, url in enumerate(retrieved_urls):
            if url in gold_urls:
                hits += 1
                sum_precisions += hits / (i + 1)
        ap = sum_precisions / num_gold

    return pd.Series([hit, mrr, ap], index=[f'Hit@{k}', f'MRR@{k}', f'MAP@{k}'])

# [Metric 2] Answer Match Accuracy (Exact Match & F1)
def normalize_answer(s):
    """
    평가를 위해 답변 텍스트를 정규화합니다.
    (소문자 변환, 문장부호 제거)
    """

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return str(text).lower()

    if not s or pd.isna(s): return ""

    return white_space_fix(remove_punc(lower(s)))

def calculate_qa_metrics(row):
    """
    정답(Answer)과 모델 예측(RAG_Answer)을 비교합니다.
    """
    gold_text = normalize_answer(row['answer'])
    pred_text = normalize_answer(row['RAG_answer'])

    # Exact Match (EM): 완전히 일치하는가? (0 or 1)
    em = 1 if gold_text == pred_text else 0


    return pd.Series([em], index=['Exact Match Accuracy'])

# ------------------------------------------------------------------
# [Execution] 전체 적용 및 저장
# ------------------------------------------------------------------
K_VALUE = 5

# 1. Retrieval Score 계산
retrieval_scores = df.apply(lambda row: calculate_retrieval_metrics(row, k=K_VALUE), axis=1)

# 2. QA Score 계산 (추가된 부분)
qa_scores = df.apply(calculate_qa_metrics, axis=1)

# 3. 전체 데이터프레임 병합
final_df = pd.concat([df, retrieval_scores, qa_scores], axis=1)

# 결과 출력 (평균 점수 확인)
print(f"======== Evaluation Results (K={K_VALUE}) ========")
# 백분율(%)로 변환하여 출력
summary = final_df[[f'Hit@{K_VALUE}', f'MRR@{K_VALUE}', f'MAP@{K_VALUE}', 'Exact Match Accuracy']].mean() * 100
print(summary)

# CSV 저장
save_path = "/content/drive/MyDrive/rag_evaluation_with_all_metrics_ny.csv"
final_df.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"\n[Done] 결과 파일이 저장되었습니다: {save_path}")

======== Evaluation Results (K=5) ========
Hit@5                   80.000000
MRR@5                   73.000000
MAP@5                   48.777778
Exact Match Accuracy    72.000000
dtype: float64

[Done] 결과 파일이 저장되었습니다: /content/drive/MyDrive/rag_evaluation_with_all_metrics_ny.csv
